In [1]:
## IMPORT LIBRARIES ----------------------------------------------------------------------------------------------------
#https://github.com/nnc-ufmg/circadipy/blob/main/src/circadipy/analysis_examples/intellicage/intellicage_analysis.ipynb
%matplotlib qt

%reload_ext autoreload
%autoreload 3

from ranking_methods import rank_accuracy
from circadipy import chrono_reader as chr  
import sys                                                                                                              # Import sys to add paths to libraries                                                                                                           # Import re to work with regular expressions
import glob                                                                                                             # Import glob to read files                                                                                                   # Import numpy to work with arrays and make calculations                                                                                            # Import time to measure time
import os                                                                                                               # Import path to work with paths                
import pandas as pd
from ranking_methods import build_all_proxies, evaluate_proxies, rank_accuracy
import numpy as np
from scipy import stats
from datetime import date, datetime, timedelta
from itertools import combinations
from scipy.stats import rankdata, spearmanr, kendalltau
import json
                                                                                      # Import pandas to work with dataframes
import warnings                                                                                                         # Import warnings to ignore warnings
warnings.filterwarnings('ignore')                                                                                       # Ignore warnings

## IMPORT CIRCADIPY ----------------------------------------------------------------------------------------------------

parent_path = os.path.dirname(os.path.dirname(os.getcwd()))
sys.path.append(parent_path)

## PCA Visualization - Dimensionality Reduction
from models import train_rf, train_gb, train_ridge, train_adaboost, train_extratrees, train_logistic
from summary import generate_summary_report
from util import  get_data_scaled, generate_features, generate_temporal_features, calculate_correlations, build_animal_protocols, get_sorted_animals_files, combine_all_features
from analysis_visualization import generate_methods_comparison, plot_animals_activity, plot_correlation, plot_pca_analysis, plot_cross_correlation, summary_visualization

In [2]:
best_feat = 'ibi_median'
#data_folder = "./data/dados_2026_06_08"
data_folder = "./data/dados_2026_05_01"
data_folder = "./data/dados_iniciais_estruturados"
dataFiles = glob.glob(f'{data_folder}/*.zip')
output_folder = './results/predict_to_predict'
ground_file = f'{data_folder}/ground.json'
os.makedirs(output_folder, exist_ok=True)
print(dataFiles)

if os.path.exists(ground_file):
    with open(ground_file, 'r') as f:
        ranks_ground = json.load(f)
        #ranks_ground = [int(f.split("_")[1]) for f in list(ranks_ground.keys())]

print(ranks_ground)
root_folder = f"{data_folder}"    



['./data/dados_iniciais_estruturados/2026-04-08 16.45.44.zip', './data/dados_iniciais_estruturados/2026-04-01 15.22.44.zip', './data/dados_iniciais_estruturados/2026-03-19 17.58.54.zip', './data/dados_iniciais_estruturados/2026-04-10 15.26.05.zip', './data/dados_iniciais_estruturados/2026-03-13 14.53.43.zip', './data/dados_iniciais_estruturados/2026-03-18 17.07.44.zip', './data/dados_iniciais_estruturados/2026-03-11 18.41.16.zip', './data/dados_iniciais_estruturados/2026-03-16 11.43.56.zip', './data/dados_iniciais_estruturados/2026-03-21 16.12.47.zip', './data/dados_iniciais_estruturados/2026-03-20 18.54.04.zip', './data/dados_iniciais_estruturados/2026-03-30 06.19.23.zip', './data/dados_iniciais_estruturados/2026-03-11 12.15.07.zip', './data/dados_iniciais_estruturados/2026-03-13 11.31.01.zip', './data/dados_iniciais_estruturados/2026-04-02 18.19.38.zip', './data/dados_iniciais_estruturados/2026-03-10 17.43.06.zip', './data/dados_iniciais_estruturados/2026-03-17 18.52.07.zip', './data

In [ ]:

for file in dataFiles:
    zip_folder = file.split("/")[-1]
    sub_folder = zip_folder.split(".")[0].replace(" ", "_")
    os.makedirs(os.path.join(root_folder, sub_folder), exist_ok=True)
    a = chr.intellicage_unwrapper([file], sub_folder, sampling_interval = '30T')



File saved in ./data/dados_iniciais_estruturados/2026-04-08_16/animal_1.txt
File saved in ./data/dados_iniciais_estruturados/2026-04-08_16/animal_10.txt
File saved in ./data/dados_iniciais_estruturados/2026-04-08_16/animal_11.txt
File saved in ./data/dados_iniciais_estruturados/2026-04-08_16/animal_12.txt
File saved in ./data/dados_iniciais_estruturados/2026-04-08_16/animal_2.txt
File saved in ./data/dados_iniciais_estruturados/2026-04-08_16/animal_3.txt
File saved in ./data/dados_iniciais_estruturados/2026-04-08_16/animal_4.txt
File saved in ./data/dados_iniciais_estruturados/2026-04-08_16/animal_5.txt
File saved in ./data/dados_iniciais_estruturados/2026-04-08_16/animal_6.txt
File saved in ./data/dados_iniciais_estruturados/2026-04-08_16/animal_7.txt
File saved in ./data/dados_iniciais_estruturados/2026-04-08_16/animal_8.txt
File saved in ./data/dados_iniciais_estruturados/2026-04-08_16/animal_9.txt
File saved in ./data/dados_iniciais_estruturados/2026-04-01_15/animal_1.txt
File save

In [3]:
individual_files = glob.glob(root_folder + "/**/*.txt", recursive=True)
individual_files = [f for f in individual_files if "animal_" in f]

start_date = date(2026, 3, 17)
end_date   = date(2026, 3, 20)
dates_to_keep = [(start_date + timedelta(days=i)).strftime("%Y-%m-%d")
                for i in range((end_date - start_date).days + 1)]


if dates_to_keep:
    individual_files = [f for f in individual_files if f.split("/")[-2].split("_")[0] in dates_to_keep]

apply_filtering = True
animals = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
animals_files = get_sorted_animals_files(individual_files, animals)
animals_protocols, animals_by_day = build_animal_protocols(animals_files, apply_filtering=apply_filtering)

Animal 1: 4
Animal 2: 4
Animal 3: 4
Animal 4: 4
Animal 5: 4
Animal 6: 4
Animal 7: 4
Animal 8: 4
Animal 9: 4
Animal 10: 4
Animal 11: 4
Animal 12: 4
Animal2 1
savgol True
Animal2 2
savgol True
Animal2 3
savgol True
Animal2 4
savgol True
Animal2 5
savgol True
Animal2 6
savgol True
Animal2 7
savgol True
Animal2 8
savgol True
Animal2 9
savgol True
Animal2 10
savgol True
Animal2 11
savgol True
Animal2 12
savgol True
Animal animal_1 Days found: 6
Animal animal_2 Days found: 6
Animal animal_3 Days found: 6
Animal animal_4 Days found: 6
Animal animal_5 Days found: 6
Animal animal_6 Days found: 6
Animal animal_7 Days found: 6
Animal animal_8 Days found: 6
Animal animal_9 Days found: 6
Animal animal_10 Days found: 6
Animal animal_11 Days found: 6
Animal animal_12 Days found: 6


In [4]:
output_path = f'{output_folder}/basic_features.csv'
features_df = generate_features(animals_protocols, output_path=output_path)

output_path = f'{output_folder}/temporal_features.csv'
temporal_df = generate_temporal_features(animals_protocols, output_path=output_path)

output_path = f'{output_folder}/all_features.csv'
all_features, feature_cols = combine_all_features(features_df, temporal_df, output_path=output_path)

X_scaled =  get_data_scaled(all_features, feature_cols)
y = []
for animal in all_features['animal'].tolist():
    y.append(ranks_ground[animal])

print(y)

all_features.head()

Saving features on ./results/predict_to_predict/basic_features.csv
Saving temporal features on ./results/predict_to_predict/temporal_features.csv
Saving all features on ./results/predict_to_predict/all_features.csv
[9, 8, 11, 10, 12, 1, 5, 6, 7, 2, 3, 4]


,animal,total_activity,mean_activity,std_activity,max_activity,min_activity,cv_activity,median_activity,max_median_ratio,activity_per_hour,...,night_ibi_cv,night_bout_len_mean,night_bout_len_cv,night_transitions_per_hour,night_onset_latency_h,night_first_2h_frac,night_gini,night_peak_hour,night_activity_per_bout,night_day_intensity_ratio
animal_1,animal_1,698.0,1.457203,2.190634,12.000000,-1.457143,1.503315,0.342857,3.500000e+01,29.083333,...,1.715948,0.441176,0.674125,1.138075,6.75,0.000000,0.000000,0.0,-0.162111,-0.011603
animal_2,animal_2,450.0,0.939457,1.626687,8.228571,-0.942857,1.731518,0.000000,8.228571e+09,18.750000,...,1.098955,0.833333,0.964365,0.602510,8.25,-0.152212,4.531877,0.0,1.352668,0.110289
animal_3,animal_3,262.0,0.546973,0.977968,5.142857,-0.857143,1.787965,0.000000,5.142857e+09,10.916667,...,1.804461,0.500000,0.718795,1.004184,6.75,-0.163968,4.528934,0.0,0.779220,0.107102
animal_4,animal_4,478.0,0.997912,1.645382,8.914286,-1.028571,1.648824,0.000000,8.914286e+09,19.916667,...,1.183685,0.600000,0.687184,0.836820,6.25,-0.521893,13.353393,0.0,0.284195,0.018263
animal_5,animal_5,416.0,0.868476,1.371375,6.171429,-1.114286,1.579060,0.000000,6.171429e+09,17.333333,...,1.245296,0.576923,0.585947,0.870293,6.75,-0.870505,21.207503,0.0,0.186526,0.017438


In [5]:
feature_rhos_path = "./data/dados_iniciais_estruturados/feature_rhos.csv"
feature_rhos_df = pd.read_csv(feature_rhos_path)

if {'feature', 'rho'}.issubset(feature_rhos_df.columns):
    feature_rhos = feature_rhos_df.set_index('feature')['rho']
else:
    raise ValueError(
        "`feature_rhos_previous.csv` must contain `feature` and `rho` columns. "
        "Recreate it with: feature_rhos.to_frame('rho').rename_axis('feature').to_csv(...)"
    )

feature_rhos.head()

feature
rhythm_ratio_24_over_harm    0.790210
power_24h                    0.706294
autocorr_12h                -0.664336
power_12h                   -0.657343
cosinor_amplitude            0.657343
Name: rho, dtype: float64

In [7]:


best_feat = 'cosinor_amplitude'
combos = [['power_24h', 'high_activity_frac', 'activity_per_bout'], ['power_24h', 'short_gap_frac', 'night_ibi_cv']]



result = []
cont = 0
for named_combo in combos:

    named_combo = combos[0]
    feature_rhos, proxies_raw = build_all_proxies(
        all_features, feature_cols, X_scaled, None,
        k=3, best_feat_idx=best_feat,
        named_combo=named_combo,
        feature_rhos=feature_rhos
    )
    for k, v in proxies_raw.items():
        print(f"{k}: {v}")



    best_feature_key = f'Best feature ({best_feat})'

    scores = proxies_raw[best_feature_key]
    pred_rank = rankdata(scores, method='ordinal')
    #print(pred_rank)
 
    if cont == 0:
        result.append({"name": best_feat, "pred": pred_rank})

    output_best_feature = f'{data_folder}/pred_{best_feat}.csv'
    #print(f"Saving output best feat {output_best_feature}")

    best_combo_key = f'Best combo ({ " + ".join(named_combo) })'
    #print(best_combo_key)
    scores = proxies_raw[best_combo_key]
    pred_rank = rankdata(scores, method='ordinal')
    #print(pred_rank)


    combo_name = "_".join(named_combo)
    result.append({"name": combo_name, "pred": pred_rank})


df = pd.DataFrame(result)
df.to_csv(f"{root_folder}/pred_data_to_predict.csv", index=False)


for r in result:
    print(r['name'])
    print(r['pred'])
    print(y)
    print(rank_accuracy(r['pred'], y))
    print()



Using feature_rhos provided externally (e.g. from a reference dataset).
Best feature (cosinor_amplitude): [0.25807574 0.24814816 0.29587971 0.27906889 0.23779534 0.1628771
 0.23798337 0.31964449 0.25556261 0.14645504 0.21094489 0.14845211]
Best combo (power_24h + high_activity_frac + activity_per_bout): [ 1.02421588  0.7916306   3.66128009  1.49751784  1.84938749 -2.20416197
 -1.31586378  0.78188016 -0.26620678 -2.15716699 -2.034411   -1.62810154]
Combo sign-aligned mean (power_24h + high_activity_frac + activity_per_bout): [ 0.34140529  0.26387687  1.2204267   0.49917261  0.6164625  -0.73472066
 -0.43862126  0.26062672 -0.08873559 -0.71905566 -0.678137   -0.54270051]
Using feature_rhos provided externally (e.g. from a reference dataset).
Best feature (cosinor_amplitude): [0.25807574 0.24814816 0.29587971 0.27906889 0.23779534 0.1628771
 0.23798337 0.31964449 0.25556261 0.14645504 0.21094489 0.14845211]
Best combo (power_24h + high_activity_frac + activity_per_bout): [ 1.02421588  0.79